# AI Governance Regulatory Spectrum Artefact v1.3 — Custom Test Notebook

## Purpose

This notebook allows researchers to apply the AI Governance Regulatory Spectrum Artefact v1.3 to their own AI-related legal, policy or strategic documents.

Unlike the replication notebook, which reproduces the thesis analysis using a fixed thesis corpus, this custom version is designed for new user-selected documents.

The model positions documents on two axes:

- X-axis: rights- and citizen-oriented safeguards ↔ innovation
- Y-axis: voluntary/soft-law ↔ mandatory/control

The same v1.3 logic is used:

- no word-level weights;
- all terms and phrases count as one occurrence;
- the document-level legal-force multiplier is applied only to the mandatory/control count.

In this custom version, document groups are expressed generically as **Group I**, **Group II**, or any other group name chosen by the researcher. The notebook does not assume that the documents belong to any specific jurisdiction.



## Step 1 — Upload and check custom PDF files

Upload the PDF documents that you want to analyse. These may be laws, bills, executive orders, policy strategies, guidance documents, frameworks, codes of conduct or other AI governance-related texts.

After uploading the files to Colab, run the following cell to check the exact filenames. The filenames printed by this cell should later be copied into the metadata table.

### Running the custom test

Upload the PDF documents you wish to analyse into the Colab session, then run the file-check cell below. Copy the exact filenames printed by the cell into the metadata template in Step 3.

The notebook is designed for two or more documents. For comparison, researchers may define groups such as **Group I** and **Group II**, but these labels can be replaced by any other categories relevant to the research design.


In [ ]:
# ============================================================
# Step 1 – Check uploaded custom PDF files
# ============================================================

import os

UPLOAD_FOLDER = "/content"

pdf_files = [
    file for file in os.listdir(UPLOAD_FOLDER)
    if file.lower().endswith(".pdf")
]

print("Number of PDF files found:", len(pdf_files))
print("\nPDF files:")

for file in sorted(pdf_files):
    print(file)


In [ ]:
# ============================================================
# Step 2A – Install required packages
# ============================================================

!pip install pymupdf spacy plotly wordcloud openpyxl
!python -m spacy download en_core_web_sm


In [ ]:
# ============================================================
# Step 2B – Import libraries
# ============================================================

import fitz
import re
import os
import pandas as pd
import spacy
import plotly.express as px
import matplotlib.pyplot as plt

from collections import Counter
from wordcloud import WordCloud

nlp = spacy.load("en_core_web_sm")

print("Libraries loaded successfully.")


In [ ]:
# ============================================================
# Step 3 – Define metadata for custom documents
# ============================================================
# Replace the example rows below with your own documents.
#
# Important:
# - file_name must exactly match the uploaded PDF filename.
# - path should normally be "/content/" + file_name.
# - legal_force_multiplier should reflect the document type.
#
# Suggested legal-force multipliers:
# Binding legislation / regulation: 1.50
# Executive order: 1.20
# Legislative proposal / bill not in force: 1.10
# Policy communication / strategy: 0.80
# Framework / guidance: 0.70
# Voluntary pact / code: 0.60
# ============================================================

documents = [
    {
        "document_id": "DOC_01",
        "document": "Your document title 1",
        "jurisdiction": "Custom",
        "plot_group": "Group I",
        "document_type": "Your document type",
        "file_name": "your_document_1.pdf",
        "path": "/content/your_document_1.pdf",
        "year": 2025,
        "institution": "Your institution",
        "legal_force_multiplier": 1.00,
        "status_assumption": "Describe the status or assumption",
        "notes": "Optional notes"
    },
    {
        "document_id": "DOC_02",
        "document": "Your document title 2",
        "jurisdiction": "Custom",
        "plot_group": "Group II",
        "document_type": "Your document type",
        "file_name": "your_document_2.pdf",
        "path": "/content/your_document_2.pdf",
        "year": 2025,
        "institution": "Your institution",
        "legal_force_multiplier": 1.00,
        "status_assumption": "Describe the status or assumption",
        "notes": "Optional notes"
    }
]

metadata_df = pd.DataFrame(documents)

print("Number of custom documents defined:", len(documents))

metadata_df[
    [
        "document_id",
        "document",
        "jurisdiction",
        "plot_group",
        "document_type",
        "file_name",
        "legal_force_multiplier"
    ]
]


### Legal-force multiplier guide

The legal-force multiplier is applied only to the mandatory/control count on the Y-axis. It does not affect the X-axis.

| Document type | Multiplier |
|---|---:|
| Binding legislation / regulation | 1.50 |
| Executive order | 1.20 |
| Legislative proposal / bill not in force | 1.10 |
| Policy communication / strategy | 0.80 |
| Framework / guidance | 0.70 |
| Voluntary pact / code | 0.60 |

Researchers may adjust these values, but any adjustment should be documented in the metadata table.


In [ ]:
# ============================================================
# Step 4 – Check whether all custom PDF files exist
# ============================================================

metadata_df["file_exists"] = metadata_df["path"].apply(lambda x: os.path.exists(x))

file_check = metadata_df[
    [
        "document_id",
        "document",
        "file_name",
        "path",
        "file_exists"
    ]
]

file_check


In [ ]:
# ============================================================
# Step 4B – Show missing custom files only
# ============================================================

missing_files = metadata_df[metadata_df["file_exists"] == False]

if len(missing_files) == 0:
    print("All custom PDF files are available.")
else:
    print("Missing files:")
    display(
        missing_files[
            [
                "document_id",
                "document",
                "file_name",
                "path"
            ]
        ]
    )


In [ ]:
# ============================================================
# Step 5 – PDF extraction and text-cleaning functions
# ============================================================

def extract_text_from_pdf(pdf_path):
    text = ""

    with fitz.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            page_text = page.get_text()
            text += f"\n\n--- Page {page_number} ---\n\n"
            text += page_text

    return text


def clean_text(text):
    # Fix hyphenated line breaks, e.g. "arti-\nficial" -> "artificial"
    text = re.sub(r"(\w+)-\s*\n\s*(\w+)", r"\1\2", text)

    text = text.lower()

    # Remove page markers and common technical artefacts
    text = re.sub(r"--- page \d+ ---", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\n\s*\d+\s*\n", " ", text)

    # Remove recurring institutional/PDF artefacts
    artefacts = [
        "federal register",
        "presidential documents",
        "official journal",
        "communication from the commission",
        "european commission",
        "brussels",
        "title 3",
        "the president"
    ]

    for item in artefacts:
        text = text.replace(item, " ")

    # Normalise whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


print("PDF extraction and cleaning functions loaded successfully.")


In [ ]:
# ============================================================
# Step 5B – Test extraction on first custom document
# ============================================================

test_doc = documents[0]

raw_text = extract_text_from_pdf(test_doc["path"])
clean = clean_text(raw_text)

print("Test document:", test_doc["document_id"], "-", test_doc["document"])
print("Raw text length:", len(raw_text))
print("Clean text length:", len(clean))
print("\nPreview:")
print(clean[:1000])


In [ ]:
# ============================================================
# Step 6 – Tokenisation, lemmatisation and customised stop-word removal
# ============================================================

generic_ai_terms = {
    "ai",
    "artificial",
    "intelligence",
    "artificial intelligence",
    "system",
    "systems",
    "model",
    "models",
    "technology",
    "technologies",
    "algorithm",
    "algorithms"
}

legal_signal_terms = {
    "shall",
    "must",
    "may",
    "should",
    "could",
    "required",
    "require",
    "requires",
    "requirement",
    "requirements"
}

noise_terms = {
    # Institutional and document artefacts
    "en", "com", "swd", "final", "european", "commission",
    "parliament", "council", "committee", "region", "brussels",
    "eu", "eur", "europe", "member", "state", "states",

    # Administrative and register artefacts
    "secretary", "agency", "order", "section", "subsection",
    "director", "day", "date", "united", "government",
    "wednesday", "november", "october", "federal", "register",
    "vol", "document", "documents", "oct", "jkt", "po", "verdate",

    # numbering and structural markers
    "ii", "iii", "iv", "paragraph", "paragraphs",
    "page", "pages", "annex", "annexes",
    "article", "articles", "art",
    "recital", "recitals",
    "chapter", "chapters",
    "title", "titles",
    "point", "points",
    "subparagraph", "subparagraphs",
    "p", "pp",

    # generic low-value processing words
    "include", "relate", "appropriate", "use"
}


def preprocess_text(text):
    doc = nlp(text)
    processed_tokens = []

    for token in doc:
        lemma = token.lemma_.lower().strip()

        # Remove punctuation, spaces and numbers
        if token.is_punct or token.is_space or token.like_num:
            continue

        # Remove very short tokens
        if len(lemma) < 2:
            continue

        # Remove generic AI terms
        if lemma in generic_ai_terms:
            continue

        # Remove corpus-specific noise terms
        if lemma in noise_terms:
            continue

        # Remove technical artefacts
        if lemma.startswith("com(") or lemma.startswith("swd("):
            continue

        if "<" in lemma or ">" in lemma:
            continue

        if re.match(r"^\d{1,2}:\d{2}$", lemma):
            continue

        # Remove stop words, except legally meaningful modal words
        if token.is_stop and lemma not in legal_signal_terms:
            continue

        # Normalise spaCy lemma for data
        if lemma == "datum":
            lemma = "data"

        processed_tokens.append(lemma)

    return processed_tokens


print("Preprocessing function loaded successfully.")


In [ ]:
# ============================================================
# Step 6B – Test preprocessing on first custom document
# ============================================================

test_tokens = preprocess_text(clean)

print("Token count:", len(test_tokens))
print("First 60 tokens:")
print(test_tokens[:60])


In [ ]:
# ============================================================
# Step 7 – v1.3 analytical vocabularies
# ============================================================

# ------------------------------------------------------------
# 7.1 Rights- and citizen-oriented safeguards vocabulary
# ------------------------------------------------------------

rights_citizen_safeguards_terms = {
    "right", "rights",
    "privacy",
    "consumer",
    "protection",
    "protect",
    "discrimination",
    "bias",
    "fairness",
    "equity",
    "equal",
    "equality",
    "confidentiality",
    "anonymisation",
    "anonymization",
    "human",

    "trustworthy",
    "responsible",
    "reliable",
    "reliability",
    "robust",
    "robustness",
    "transparent",
    "transparency",
    "accountable",
    "accountability",
    "explainable",
    "explainability",
    "oversight",
    "supervision",
    "monitoring",
    "audit",
    "auditing",
    "assessment",
    "evaluation",
    "certification",
    "conformity",
    "compliance",
    "liability",
    "integrity",
    "safety",
    "safe",
    "harm",
    "risk"
}

rights_citizen_safeguards_phrases = {
    "fundamental rights",
    "human rights",
    "civil rights",
    "data protection",
    "personal data",
    "consumer protection",
    "privacy protection",
    "non-discrimination",
    "algorithmic discrimination",
    "algorithmic bias",
    "human oversight",
    "human control",
    "human agency",
    "fundamental rights impact assessment",
    "affected persons",
    "natural persons",
    "rights and freedoms",

    "trustworthy ai",
    "responsible ai",
    "risk management",
    "risk assessment",
    "risk mitigation",
    "risk-based approach",
    "high risk",
    "high-risk",
    "systemic risk",
    "conformity assessment",
    "impact assessment",
    "bias mitigation",
    "societal harms",
    "public safety",
    "transparency obligations",
    "accountability mechanisms",
    "risk management framework",
    "safe and secure",
    "safe, secure"
}


# ------------------------------------------------------------
# 7.2 Innovation vocabulary
# ------------------------------------------------------------

innovation_terms = {
    "innovation",
    "innovative",
    "innovate",
    "deployment",
    "deploy",
    "adoption",
    "uptake",
    "experimentation",
    "experiment",
    "sandbox",
    "prototype",
    "piloting",
    "pilot",
    "scale",
    "scaling",

    "competitiveness",
    "competitive",
    "competition",
    "growth",
    "investment",
    "invest",
    "entrepreneurship",
    "business",
    "industry",
    "industrial",
    "commercial",
    "productivity",
    "productive",
    "prosperity",
    "economic",
    "economy",

    "research",
    "science",
    "scientific",
    "skill",
    "skills",
    "education",
    "talent",
    "digital",
    "transformation",
    "modernisation",
    "modernization",

    "leadership",
    "advance",
    "advancement",
    "value",
    "efficiency",
    "excellence",
    "ecosystem"
}

innovation_phrases = {
    "market uptake",
    "digital transformation",
    "economic growth",
    "technological leadership",
    "innovation leadership",
    "industrial strategy",
    "industrial competitiveness",
    "value creation",
    "research and innovation",
    "business opportunities",
    "competitive advantage",
    "innovation ecosystem",
    "skills development",
    "talent development",
    "workforce development",
    "public sector innovation",
    "private sector innovation",
    "regulatory sandbox",
    "regulatory sandboxes",
    "testing environment",
    "real-world testing",
    "technology transfer",
    "scientific progress",
    "economic opportunity",
    "market development",
    "investment in ai",
    "digital economy",
    "productivity growth",
    "innovation capacity",
    "global leadership"
}


# ------------------------------------------------------------
# 7.3 Mandatory/control vocabulary
# ------------------------------------------------------------
# v1.3 logic:
# - no word-level weights;
# - broad legal-context terms excluded:
#   law, regulation, rule, rules, authority, authorities;
# - implementation/control terms retained:
#   monitoring, supervision, supervise, market surveillance,
#   ensure, ensures, ensuring, liability.
# ------------------------------------------------------------

mandatory_terms = {
    "shall",
    "must",
    "obliged",
    "obligation",
    "obligations",
    "mandatory",
    "binding",

    "required",
    "require",
    "requires",
    "requirement",
    "requirements",

    "prohibit",
    "prohibited",
    "prohibition",
    "ban",
    "banned",

    "comply",
    "compliance",
    "enforce",
    "enforcement",

    "penalty",
    "penalties",
    "sanction",
    "sanctions",
    "fine",
    "fines",

    "monitoring",
    "supervision",
    "supervise",
    "liability",
    "ensure",
    "ensures",
    "ensuring"
}

mandatory_phrases = {
    "legal obligation",
    "legal obligations",
    "binding obligation",
    "binding obligations",
    "binding requirement",
    "binding requirements",
    "mandatory requirement",
    "mandatory requirements",

    "shall ensure",
    "shall develop",
    "shall submit",
    "shall establish",
    "shall provide",
    "shall require",
    "shall implement",
    "shall take",

    "must comply",
    "required to",

    "compliance with",
    "enforcement action",
    "subject to",
    "in accordance with",
    "pursuant to",
    "regulatory requirement",
    "regulatory requirements",

    "market surveillance",
    "administrative fine",
    "administrative fines"
}


# ------------------------------------------------------------
# 7.4 Voluntary / soft-law vocabulary
# ------------------------------------------------------------

voluntary_terms = {
    "may",
    "should",
    "could",

    "encourage",
    "encourages",
    "encouraged",

    "recommend",
    "recommends",
    "recommendation",
    "recommendations",

    "guidance",
    "guideline",
    "guidelines",

    "framework",
    "frameworks",

    "principle",
    "principles",

    "practice",
    "practices",

    "voluntary",
    "optional",

    "support",
    "supports",
    "promote",
    "promotes",
    "facilitate",
    "facilitates",
    "enable",
    "enables",
    "foster",
    "fosters",

    "cooperation",
    "collaboration",
    "coordination",
    "partnership",

    "consultation",
    "stakeholder",
    "stakeholders"
}

voluntary_phrases = {
    "best practice",
    "best practices",

    "voluntary commitment",
    "voluntary commitments",
    "voluntary framework",

    "non mandatory",
    "non-mandatory",
    "not mandatory",

    "non binding",
    "non-binding",
    "not binding",

    "not required",
    "no obligation",
    "without obligation",

    "guidance document",
    "guidance documents",

    "code of conduct",
    "codes of conduct",

    "soft law",

    "should consider",
    "may include",
    "may consider",

    "encourage the",
    "encourages the",

    "public consultation",
    "stakeholder engagement",
    "stakeholder consultation",

    "public-private partnership",
    "international cooperation",
    "technical guidance",
    "voluntary guidelines",
    "common framework",
    "risk management framework"
}

print("v1.3 custom-corpus vocabularies loaded successfully.")
print("Rights/citizen terms:", len(rights_citizen_safeguards_terms))
print("Rights/citizen phrases:", len(rights_citizen_safeguards_phrases))
print("Innovation terms:", len(innovation_terms))
print("Innovation phrases:", len(innovation_phrases))
print("Mandatory terms:", len(mandatory_terms))
print("Mandatory phrases:", len(mandatory_phrases))
print("Voluntary terms:", len(voluntary_terms))
print("Voluntary phrases:", len(voluntary_phrases))


In [ ]:
# ============================================================
# Step 8 – Counting, phrase-priority and scoring functions
# ============================================================

def count_phrases_in_text(clean_text, phrase_set):
    total = 0
    phrase_counts = {}

    for phrase in phrase_set:
        pattern = r"\b" + re.escape(phrase) + r"\b"
        count = len(re.findall(pattern, clean_text))

        if count > 0:
            phrase_counts[phrase] = count
            total += count

    return total, phrase_counts


def phrase_to_components(phrase):
    phrase = phrase.replace("-", " ")
    doc = nlp(phrase)
    components = []

    for token in doc:
        lemma = token.lemma_.lower().strip()

        if token.is_punct or token.is_space:
            continue

        if lemma in {"and", "or", "the", "of", "to", "in", "with"}:
            continue

        if lemma == "datum":
            lemma = "data"

        components.append(lemma)

    return components


def remove_phrase_components_from_tokens(tokens, phrase_details):
    tokens_copy = tokens.copy()

    for phrase, count in phrase_details.items():
        components = phrase_to_components(phrase)

        for _ in range(count):
            for word in components:
                if word in tokens_copy:
                    tokens_copy.remove(word)

    return tokens_copy


def count_terms(tokens, dictionary):
    counter = Counter(tokens)
    total = 0

    for term in dictionary:
        if " " not in term:
            total += counter[term]

    return total


def compute_axis_score(positive_count, negative_count):
    denominator = positive_count + negative_count

    if denominator == 0:
        return 0

    return ((positive_count - negative_count) / denominator) * 100


def compute_adjusted_y_score(mandatory_count, voluntary_count, legal_force_multiplier):
    adjusted_mandatory = mandatory_count * legal_force_multiplier
    denominator = adjusted_mandatory + voluntary_count

    if denominator == 0:
        return 0

    return ((adjusted_mandatory - voluntary_count) / denominator) * 100


print("Counting and scoring functions loaded successfully.")

In [ ]:
# ============================================================
# Step 9 – Main document-analysis function for v1.3
# ============================================================

def analyse_document_v1_3(tokens, clean_text, legal_force_multiplier=1.0):
    # 1. Count multi-word expressions first
    rights_phrase, rights_phrase_details = count_phrases_in_text(
        clean_text,
        rights_citizen_safeguards_phrases
    )

    innovation_phrase, innovation_phrase_details = count_phrases_in_text(
        clean_text,
        innovation_phrases
    )

    mandatory_phrase, mandatory_phrase_details = count_phrases_in_text(
        clean_text,
        mandatory_phrases
    )

    voluntary_phrase, voluntary_phrase_details = count_phrases_in_text(
        clean_text,
        voluntary_phrases
    )

    # 2. Remove phrase components to reduce double-counting
    all_phrase_details = {}
    for phrase_dict in [
        rights_phrase_details,
        innovation_phrase_details,
        mandatory_phrase_details,
        voluntary_phrase_details
    ]:
        all_phrase_details.update(phrase_dict)

    tokens_without_phrase_components = remove_phrase_components_from_tokens(
        tokens,
        all_phrase_details
    )

    # 3. Count single-word terms
    rights_single = count_terms(
        tokens_without_phrase_components,
        rights_citizen_safeguards_terms
    )

    innovation_single = count_terms(
        tokens_without_phrase_components,
        innovation_terms
    )

    mandatory_single = count_terms(
        tokens_without_phrase_components,
        mandatory_terms
    )

    voluntary_single = count_terms(
        tokens_without_phrase_components,
        voluntary_terms
    )

    # 4. Combine phrase and single-word counts
    rights_total = rights_single + rights_phrase
    innovation_total = innovation_single + innovation_phrase
    mandatory_total = mandatory_single + mandatory_phrase
    voluntary_total = voluntary_single + voluntary_phrase

    # 5. Compute X and Y scores
    x_score = compute_axis_score(
        positive_count=innovation_total,
        negative_count=rights_total
    )

    y_score_adjusted = compute_adjusted_y_score(
        mandatory_count=mandatory_total,
        voluntary_count=voluntary_total,
        legal_force_multiplier=legal_force_multiplier
    )

    # 6. Relative frequencies per 1,000 tokens
    if len(tokens) == 0:
        rights_per_1000 = 0
        innovation_per_1000 = 0
        mandatory_per_1000 = 0
        voluntary_per_1000 = 0
    else:
        rights_per_1000 = round((rights_total / len(tokens)) * 1000, 2)
        innovation_per_1000 = round((innovation_total / len(tokens)) * 1000, 2)
        mandatory_per_1000 = round((mandatory_total / len(tokens)) * 1000, 2)
        voluntary_per_1000 = round((voluntary_total / len(tokens)) * 1000, 2)

    return {
        "total_tokens": len(tokens),

        "rights_citizen_safeguards_single": rights_single,
        "rights_citizen_safeguards_phrases": rights_phrase,
        "rights_citizen_safeguards_total": rights_total,

        "innovation_single": innovation_single,
        "innovation_phrases": innovation_phrase,
        "innovation_total": innovation_total,

        "mandatory_single": mandatory_single,
        "mandatory_phrases": mandatory_phrase,
        "mandatory_total": mandatory_total,

        "voluntary_single": voluntary_single,
        "voluntary_phrases": voluntary_phrase,
        "voluntary_total": voluntary_total,

        "rights_citizen_safeguards_per_1000": rights_per_1000,
        "innovation_per_1000": innovation_per_1000,
        "mandatory_per_1000": mandatory_per_1000,
        "voluntary_per_1000": voluntary_per_1000,

        "x_score": round(x_score, 2),
        "y_score_adjusted": round(y_score_adjusted, 2),

        "rights_phrase_details": rights_phrase_details,
        "innovation_phrase_details": innovation_phrase_details,
        "mandatory_phrase_details": mandatory_phrase_details,
        "voluntary_phrase_details": voluntary_phrase_details
    }


print("Main v1.3 document-analysis function loaded successfully.")

In [ ]:
# ============================================================
# Step 10 – Text-extraction quality control for custom corpus
# ============================================================

quality_rows = []

for doc_info in documents:
    try:
        raw_text = extract_text_from_pdf(doc_info["path"])
        clean = clean_text(raw_text)
        tokens = preprocess_text(clean)

        quality_rows.append({
            "document_id": doc_info.get("document_id"),
            "document": doc_info["document"],
            "jurisdiction": doc_info["jurisdiction"],
            "plot_group": doc_info.get("plot_group", doc_info["jurisdiction"]),
            "document_type": doc_info["document_type"],
            "raw_text_length": len(raw_text),
            "clean_text_length": len(clean),
            "token_count": len(tokens),
            "status": "OK"
        })

    except Exception as e:
        quality_rows.append({
            "document_id": doc_info.get("document_id"),
            "document": doc_info.get("document"),
            "jurisdiction": doc_info.get("jurisdiction"),
            "plot_group": doc_info.get("plot_group", doc_info.get("jurisdiction")),
            "document_type": doc_info.get("document_type"),
            "raw_text_length": None,
            "clean_text_length": None,
            "token_count": None,
            "status": str(e)
        })

quality_df = pd.DataFrame(quality_rows)

quality_df


In [ ]:
# ============================================================
# Step 10B – Inspect shortest custom-corpus documents
# ============================================================

quality_df.sort_values("token_count")[
    [
        "document_id",
        "document",
        "jurisdiction",
        "plot_group",
        "document_type",
        "token_count",
        "status"
    ]
]


In [ ]:
# ============================================================
# Step 11 – Run custom v1.3 corpus analysis
# ============================================================

final_rows = []

for doc_info in documents:
    print(f"Processing: {doc_info['document_id']} - {doc_info['document']}")

    raw_text = extract_text_from_pdf(doc_info["path"])
    clean = clean_text(raw_text)
    tokens = preprocess_text(clean)

    result = analyse_document_v1_3(
        tokens,
        clean,
        legal_force_multiplier=doc_info["legal_force_multiplier"]
    )

    row = {
        "document_id": doc_info.get("document_id"),
        "document": doc_info["document"],
        "jurisdiction": doc_info["jurisdiction"],
        "plot_group": doc_info.get("plot_group", doc_info["jurisdiction"]),
        "document_type": doc_info["document_type"],
        "year": doc_info.get("year"),
        "institution": doc_info.get("institution"),
        "legal_force_multiplier": doc_info["legal_force_multiplier"],
        "status_assumption": doc_info.get("status_assumption"),
        "notes": doc_info.get("notes"),
        **result
    }

    final_rows.append(row)

final_results_df = pd.DataFrame(final_rows)

final_results_df[
    [
        "document_id",
        "document",
        "jurisdiction",
        "plot_group",
        "document_type",
        "total_tokens",
        "rights_citizen_safeguards_total",
        "innovation_total",
        "mandatory_total",
        "voluntary_total",
        "rights_citizen_safeguards_per_1000",
        "innovation_per_1000",
        "mandatory_per_1000",
        "voluntary_per_1000",
        "x_score",
        "y_score_adjusted"
    ]
]


In [ ]:
# ============================================================
# Step 11B – Clean custom summary table
# ============================================================

final_results_summary = final_results_df[
    [
        "document_id",
        "document",
        "jurisdiction",
        "plot_group",
        "document_type",
        "year",
        "total_tokens",
        "rights_citizen_safeguards_per_1000",
        "innovation_per_1000",
        "mandatory_per_1000",
        "voluntary_per_1000",
        "x_score",
        "y_score_adjusted"
    ]
].copy()

final_results_summary.sort_values("document_id")


In [ ]:
# ============================================================
# Step 12 – Custom-corpus regulatory spectrum graph
# ============================================================

fig_full = px.scatter(
    final_results_df,
    x="x_score",
    y="y_score_adjusted",
    color="plot_group",
    symbol="document_type",
    text="document_id",
    hover_data=[
        "document",
        "document_type",
        "year",
        "institution",
        "legal_force_multiplier",
        "rights_citizen_safeguards_per_1000",
        "innovation_per_1000",
        "mandatory_per_1000",
        "voluntary_per_1000"
    ],
    title="Custom Corpus v1.3: Rights- and Citizen-Oriented Safeguards versus Innovation",
    labels={
        "x_score": "Rights- and citizen-oriented safeguards (-100) ↔ Innovation (+100)",
        "y_score_adjusted": "Voluntary (-100) ↔ Mandatory (+100)",
        "plot_group": "Document group",
        "document_type": "Document type"
    }
)

fig_full.add_vline(x=0, line_dash="dash")
fig_full.add_hline(y=0, line_dash="dash")

fig_full.add_annotation(
    x=-65,
    y=75,
    text="Mandatory rights/citizen safeguards",
    showarrow=False
)

fig_full.add_annotation(
    x=65,
    y=75,
    text="Mandatory innovation",
    showarrow=False
)

fig_full.add_annotation(
    x=-65,
    y=-75,
    text="Voluntary rights/citizen safeguards",
    showarrow=False
)

fig_full.add_annotation(
    x=65,
    y=-75,
    text="Voluntary innovation",
    showarrow=False
)

fig_full.update_traces(textposition="top center")

fig_full.update_layout(
    xaxis=dict(range=[-100, 100]),
    yaxis=dict(range=[-100, 100]),
    width=1350,
    height=900
)

fig_full.show()


In [ ]:
# ============================================================
# Step 13 – Custom four-component vocabulary profile
# ============================================================

four_component_full = final_results_df[
    [
        "document_id",
        "document",
        "jurisdiction",
        "plot_group",
        "document_type",
        "rights_citizen_safeguards_per_1000",
        "innovation_per_1000",
        "mandatory_per_1000",
        "voluntary_per_1000"
    ]
].copy()

four_component_long_full = four_component_full.melt(
    id_vars=["document_id", "document", "jurisdiction", "plot_group", "document_type"],
    value_vars=[
        "rights_citizen_safeguards_per_1000",
        "innovation_per_1000",
        "mandatory_per_1000",
        "voluntary_per_1000"
    ],
    var_name="component",
    value_name="frequency_per_1000_tokens"
)

component_labels = {
    "rights_citizen_safeguards_per_1000": "Rights/citizen safeguards",
    "innovation_per_1000": "Innovation",
    "mandatory_per_1000": "Mandatory/control",
    "voluntary_per_1000": "Voluntary/soft-law"
}

four_component_long_full["component"] = four_component_long_full["component"].map(component_labels)

fig_components_full = px.bar(
    four_component_long_full,
    x="document_id",
    y="frequency_per_1000_tokens",
    color="component",
    barmode="group",
    hover_data=["document", "jurisdiction", "document_type"],
    title="Custom Corpus v1.3: Four-Component Vocabulary Profile",
    labels={
        "document_id": "Document ID",
        "frequency_per_1000_tokens": "Detected terms per 1,000 tokens",
        "component": "Analytical component"
    }
)

fig_components_full.update_layout(
    width=1500,
    height=750
)

fig_components_full.show()


In [ ]:
# ============================================================
# Step 14 – Analytical vocabulary frequency table v1.3
# ============================================================

analytical_rows = []

for doc_info in documents:
    print(f"Building vocabulary table for: {doc_info['document_id']} - {doc_info['document']}")

    raw_text = extract_text_from_pdf(doc_info["path"])
    clean = clean_text(raw_text)
    tokens = preprocess_text(clean)
    total_tokens = len(tokens)

    token_counter = Counter(tokens)

    base_info = {
        "document_id": doc_info.get("document_id"),
        "document": doc_info.get("document"),
        "jurisdiction": doc_info.get("jurisdiction"),
        "plot_group": doc_info.get("plot_group", doc_info.get("jurisdiction")),
        "document_type": doc_info.get("document_type"),
        "total_tokens": total_tokens
    }

    # Rights/citizen safeguards: single words
    for term in rights_citizen_safeguards_terms:
        count = token_counter[term]
        if count > 0:
            analytical_rows.append({
                **base_info,
                "category": "rights_citizen_safeguards",
                "term": term,
                "term_type": "single_word",
                "absolute_frequency": count,
                "frequency_per_1000": round((count / total_tokens) * 1000, 3)
            })

    # Rights/citizen safeguards: phrases
    for phrase in rights_citizen_safeguards_phrases:
        pattern = r"\b" + re.escape(phrase) + r"\b"
        count = len(re.findall(pattern, clean))
        if count > 0:
            analytical_rows.append({
                **base_info,
                "category": "rights_citizen_safeguards",
                "term": phrase,
                "term_type": "multi_word_expression",
                "absolute_frequency": count,
                "frequency_per_1000": round((count / total_tokens) * 1000, 3)
            })

    # Innovation: single words
    for term in innovation_terms:
        count = token_counter[term]
        if count > 0:
            analytical_rows.append({
                **base_info,
                "category": "innovation",
                "term": term,
                "term_type": "single_word",
                "absolute_frequency": count,
                "frequency_per_1000": round((count / total_tokens) * 1000, 3)
            })

    # Innovation: phrases
    for phrase in innovation_phrases:
        pattern = r"\b" + re.escape(phrase) + r"\b"
        count = len(re.findall(pattern, clean))
        if count > 0:
            analytical_rows.append({
                **base_info,
                "category": "innovation",
                "term": phrase,
                "term_type": "multi_word_expression",
                "absolute_frequency": count,
                "frequency_per_1000": round((count / total_tokens) * 1000, 3)
            })

    # Mandatory/control: single words
    for term in mandatory_terms:
        count = token_counter[term]
        if count > 0:
            analytical_rows.append({
                **base_info,
                "category": "mandatory_control",
                "term": term,
                "term_type": "single_word",
                "absolute_frequency": count,
                "frequency_per_1000": round((count / total_tokens) * 1000, 3)
            })

    # Mandatory/control: phrases
    for phrase in mandatory_phrases:
        pattern = r"\b" + re.escape(phrase) + r"\b"
        count = len(re.findall(pattern, clean))
        if count > 0:
            analytical_rows.append({
                **base_info,
                "category": "mandatory_control",
                "term": phrase,
                "term_type": "multi_word_expression",
                "absolute_frequency": count,
                "frequency_per_1000": round((count / total_tokens) * 1000, 3)
            })

    # Voluntary/soft-law: single words
    for term in voluntary_terms:
        count = token_counter[term]
        if count > 0:
            analytical_rows.append({
                **base_info,
                "category": "voluntary_soft_law",
                "term": term,
                "term_type": "single_word",
                "absolute_frequency": count,
                "frequency_per_1000": round((count / total_tokens) * 1000, 3)
            })

    # Voluntary/soft-law: phrases
    for phrase in voluntary_phrases:
        pattern = r"\b" + re.escape(phrase) + r"\b"
        count = len(re.findall(pattern, clean))
        if count > 0:
            analytical_rows.append({
                **base_info,
                "category": "voluntary_soft_law",
                "term": phrase,
                "term_type": "multi_word_expression",
                "absolute_frequency": count,
                "frequency_per_1000": round((count / total_tokens) * 1000, 3)
            })


analytical_vocabulary_df = pd.DataFrame(analytical_rows)

analytical_vocabulary_df = analytical_vocabulary_df.sort_values(
    by=["document_id", "category", "absolute_frequency"],
    ascending=[True, True, False]
)

analytical_vocabulary_df.head(30)

In [ ]:
# ============================================================
# Step 14B – Top 20 analytical terms per document and category
# ============================================================

top20_analytical_vocabulary_df = (
    analytical_vocabulary_df
    .groupby(["document_id", "category"], group_keys=False)
    .head(20)
    .reset_index(drop=True)
)

top20_analytical_vocabulary_df[
    [
        "document_id",
        "document",
        "category",
        "term",
        "term_type",
        "absolute_frequency",
        "frequency_per_1000"
    ]
]

In [ ]:
# ============================================================
# Step 15 – Analytical word clouds by document group
# ============================================================

def generate_wordcloud_from_frequency_dict(freq_dict, title):
    wordcloud = WordCloud(
        width=1200,
        height=700,
        background_color="white",
        max_words=120,
        collocations=False
    ).generate_from_frequencies(freq_dict)

    plt.figure(figsize=(14, 8))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(title, fontsize=18)
    plt.show()


# Generate one analytical word cloud for each user-defined document group.
for group_name in sorted(analytical_vocabulary_df["plot_group"].dropna().unique()):
    group_terms = analytical_vocabulary_df[
        analytical_vocabulary_df["plot_group"] == group_name
    ]

    group_freq = (
        group_terms
        .groupby("term")["absolute_frequency"]
        .sum()
        .to_dict()
    )

    if len(group_freq) > 0:
        generate_wordcloud_from_frequency_dict(
            group_freq,
            f"{group_name} – analytical vocabulary word cloud"
        )
    else:
        print(f"No analytical vocabulary terms found for {group_name}.")


In [ ]:
# ============================================================
# Step 16 – Heatmap of four-component intensity
# ============================================================

heatmap_df = final_results_df[
    [
        "document_id",
        "rights_citizen_safeguards_per_1000",
        "innovation_per_1000",
        "mandatory_per_1000",
        "voluntary_per_1000"
    ]
].copy()

heatmap_df = heatmap_df.set_index("document_id")

heatmap_df.columns = [
    "Rights/citizen safeguards",
    "Innovation",
    "Mandatory/control",
    "Voluntary/soft-law"
]

fig_heatmap = px.imshow(
    heatmap_df,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="Blues",
    title="Custom Corpus v1.3: Heatmap of Four-Component Intensity",
    labels=dict(
        x="Analytical component",
        y="Document ID",
        color="Frequency per 1,000 tokens"
    )
)

fig_heatmap.update_layout(
    width=1100,
    height=900
)

fig_heatmap.show()


In [ ]:
# ============================================================
# Step 16 – Average position plot by document group
# ============================================================

average_position_df = (
    final_results_df
    .groupby("plot_group")
    .agg(
        mean_x_score=("x_score", "mean"),
        mean_y_score=("y_score_adjusted", "mean"),
        std_x_score=("x_score", "std"),
        std_y_score=("y_score_adjusted", "std"),
        document_count=("document_id", "count")
    )
    .reset_index()
)

average_position_df["mean_x_score"] = average_position_df["mean_x_score"].round(2)
average_position_df["mean_y_score"] = average_position_df["mean_y_score"].round(2)
average_position_df["std_x_score"] = average_position_df["std_x_score"].round(2)
average_position_df["std_y_score"] = average_position_df["std_y_score"].round(2)

average_position_df

In [ ]:
# ============================================================
# Step 18B – Visualise average positions
# ============================================================

fig_average = px.scatter(
    average_position_df,
    x="mean_x_score",
    y="mean_y_score",
    color="plot_group",
    size="document_count",
    text="plot_group",
    error_x="std_x_score",
    error_y="std_y_score",
    title="Custom Corpus v1.3: Average Regulatory Position by Document Group",
    labels={
        "mean_x_score": "Rights/citizen safeguards (-100) ↔ Innovation (+100)",
        "mean_y_score": "Voluntary (-100) ↔ Mandatory (+100)",
        "plot_group": "Document group",
        "document_count": "Number of documents"
    }
)

fig_average.add_vline(x=0, line_dash="dash")
fig_average.add_hline(y=0, line_dash="dash")

fig_average.update_traces(textposition="top center")

fig_average.update_layout(
    xaxis=dict(range=[-100, 100]),
    yaxis=dict(range=[-100, 100]),
    width=1100,
    height=800
)

fig_average.show()


In [ ]:
# ============================================================
# Step 16 – Assign each document to a regulatory quadrant
# ============================================================

def classify_quadrant(x, y):
    if x < 0 and y >= 0:
        return "Mandatory rights/citizen safeguards"
    elif x >= 0 and y >= 0:
        return "Mandatory innovation"
    elif x < 0 and y < 0:
        return "Voluntary rights/citizen safeguards"
    else:
        return "Voluntary innovation"

quadrant_df = final_results_df.copy()
quadrant_df["quadrant"] = quadrant_df.apply(
    lambda row: classify_quadrant(row["x_score"], row["y_score_adjusted"]),
    axis=1
)

quadrant_df[
    [
        "document_id",
        "document",
        "plot_group",
        "x_score",
        "y_score_adjusted",
        "quadrant"
    ]
].head(10)

In [ ]:
# ============================================================
# Step 19B – Count documents by quadrant and group
# ============================================================

quadrant_summary = (
    quadrant_df
    .groupby(["plot_group", "quadrant"])
    .size()
    .reset_index(name="document_count")
)

quadrant_summary

In [ ]:
# ============================================================
# Step 19C – Quadrant distribution chart
# ============================================================

fig_quadrant = px.bar(
    quadrant_summary,
    x="quadrant",
    y="document_count",
    color="plot_group",
    barmode="group",
    text="document_count",
    title="Custom Corpus v1.3: Distribution of Documents Across Regulatory Quadrants",
    labels={
        "quadrant": "Regulatory quadrant",
        "document_count": "Number of documents",
        "plot_group": "Document group"
    }
)

fig_quadrant.update_traces(textposition="outside")
fig_quadrant.update_layout(
    width=1300,
    height=750,
    xaxis_tickangle=-20
)

fig_quadrant.show()


In [ ]:
# ============================================================
# Step 16 – Top driver terms by document group
# ============================================================

group_token_totals = (
    final_results_df
    .groupby("plot_group")["total_tokens"]
    .sum()
    .to_dict()
)

driver_terms_df = (
    analytical_vocabulary_df
    .groupby(["plot_group", "term", "category"], as_index=False)
    .agg(absolute_frequency=("absolute_frequency", "sum"))
)

driver_terms_df["group_total_tokens"] = driver_terms_df["plot_group"].map(group_token_totals)

driver_terms_df["corpus_frequency_per_1000"] = (
    driver_terms_df["absolute_frequency"] /
    driver_terms_df["group_total_tokens"]
) * 1000

driver_terms_df["corpus_frequency_per_1000"] = driver_terms_df["corpus_frequency_per_1000"].round(3)

top_driver_terms_df = (
    driver_terms_df
    .sort_values(["plot_group", "corpus_frequency_per_1000"], ascending=[True, False])
    .groupby("plot_group", group_keys=False)
    .head(15)
    .reset_index(drop=True)
)

top_driver_terms_df


In [ ]:
# ============================================================
# Step 20B – Visualise top driver terms by document group
# ============================================================

plot_top_driver_terms_df = top_driver_terms_df.sort_values(
    ["plot_group", "corpus_frequency_per_1000"],
    ascending=[True, True]
)

fig_driver_terms = px.bar(
    plot_top_driver_terms_df,
    x="corpus_frequency_per_1000",
    y="term",
    color="category",
    facet_col="plot_group",
    orientation="h",
    hover_data=["absolute_frequency"],
    title="Custom Corpus v1.3: Top Driver Terms by Document Group",
    labels={
        "corpus_frequency_per_1000": "Frequency per 1,000 group tokens",
        "term": "Term / phrase",
        "category": "Analytical category",
        "plot_group": "Document group"
    }
)

fig_driver_terms.update_layout(
    width=1400,
    height=800
)

fig_driver_terms.show()


In [ ]:
# ============================================================
# Step 16 – Slope chart: rights/citizen safeguards vs innovation
# ============================================================

slope_df = final_results_df[
    [
        "document_id",
        "document",
        "jurisdiction",
        "plot_group",
        "document_type",
        "rights_citizen_safeguards_per_1000",
        "innovation_per_1000"
    ]
].copy()

slope_long_df = slope_df.melt(
    id_vars=["document_id", "document", "jurisdiction", "plot_group", "document_type"],
    value_vars=["rights_citizen_safeguards_per_1000", "innovation_per_1000"],
    var_name="component",
    value_name="frequency_per_1000"
)

slope_component_labels = {
    "rights_citizen_safeguards_per_1000": "Rights/citizen safeguards",
    "innovation_per_1000": "Innovation"
}

slope_long_df["component"] = slope_long_df["component"].map(slope_component_labels)

fig_slope = px.line(
    slope_long_df,
    x="component",
    y="frequency_per_1000",
    line_group="document_id",
    color="plot_group",
    hover_data=["document_id", "document", "document_type"],
    markers=True,
    title="Custom Corpus v1.3: Slope Chart of Rights/Citizen Safeguards versus Innovation",
    labels={
        "component": "Analytical component",
        "frequency_per_1000": "Detected terms per 1,000 tokens",
        "plot_group": "Document group"
    }
)

fig_slope.update_layout(
    width=1200,
    height=800
)

fig_slope.show()


In [ ]:
# ============================================================
# Step 21B – Prepare Group I and Group II slope charts
# ============================================================
# These charts are optional. They are useful when the custom corpus uses
# the default groups "Group I" and "Group II". If those group names are
# replaced by the researcher, this cell will simply report that the group
# was not found.

def build_group_slope_chart(group_name):
    group_slope_long_df = slope_long_df[
        slope_long_df["plot_group"] == group_name
    ]

    if group_slope_long_df.empty:
        print(f"No documents found for {group_name}.")
        return None

    fig = px.line(
        group_slope_long_df,
        x="component",
        y="frequency_per_1000",
        line_group="document_id",
        color="document_id",
        hover_data=["document", "document_type"],
        markers=True,
        title=f"{group_name}: Rights/Citizen Safeguards versus Innovation",
        labels={
            "component": "Analytical component",
            "frequency_per_1000": "Detected terms per 1,000 tokens",
            "document_id": "Document ID"
        }
    )

    fig.update_layout(
        width=1000,
        height=750
    )

    fig.show()
    return fig


fig_slope_group_i = build_group_slope_chart("Group I")
fig_slope_group_ii = build_group_slope_chart("Group II")


In [ ]:
# ============================================================
# Step 16 – Document-type comparison chart
# ============================================================

document_type_summary = (
    final_results_df
    .groupby("document_type")
    .agg(
        mean_x_score=("x_score", "mean"),
        mean_y_score=("y_score_adjusted", "mean"),
        document_count=("document_id", "count")
    )
    .reset_index()
)

document_type_summary["mean_x_score"] = document_type_summary["mean_x_score"].round(2)
document_type_summary["mean_y_score"] = document_type_summary["mean_y_score"].round(2)

document_type_summary

In [ ]:
# ============================================================
# Step 22B – Visualise average scores by document type
# ============================================================

document_type_long = document_type_summary.melt(
    id_vars=["document_type", "document_count"],
    value_vars=["mean_x_score", "mean_y_score"],
    var_name="metric",
    value_name="average_score"
)

metric_labels = {
    "mean_x_score": "Average X-score",
    "mean_y_score": "Average Y-score"
}

document_type_long["metric"] = document_type_long["metric"].map(metric_labels)

fig_doc_type = px.bar(
    document_type_long,
    x="average_score",
    y="document_type",
    color="metric",
    orientation="h",
    barmode="group",
    hover_data=["document_count"],
    title="Custom Corpus v1.3: Average Regulatory Position by Document Type",
    labels={
        "average_score": "Average score",
        "document_type": "Document type",
        "metric": "Score type",
        "document_count": "Number of documents"
    }
)

fig_doc_type.add_vline(x=0, line_dash="dash")

fig_doc_type.update_layout(
    width=1400,
    height=850
)

fig_doc_type.show()


In [ ]:
# ============================================================
# Step 16 – Export custom analysis outputs
# ============================================================

# Main tables
final_results_df.to_excel(
    "/content/custom_corpus_results_v1_3.xlsx",
    index=False
)

final_results_summary.to_excel(
    "/content/custom_corpus_summary_v1_3.xlsx",
    index=False
)

analytical_vocabulary_df.to_excel(
    "/content/custom_analytical_vocabulary_table_v1_3.xlsx",
    index=False
)

top20_analytical_vocabulary_df.to_excel(
    "/content/custom_top20_analytical_vocabulary_by_document_v1_3.xlsx",
    index=False
)

# Core visualisations
fig_full.write_html(
    "/content/custom_regulatory_spectrum_v1_3.html"
)

fig_components_full.write_html(
    "/content/custom_four_component_profile_v1_3.html"
)

# Optional explanatory visualisations
fig_heatmap.write_html(
    "/content/custom_heatmap_v1_3.html"
)

fig_average.write_html(
    "/content/custom_average_position_v1_3.html"
)

fig_quadrant.write_html(
    "/content/custom_quadrant_distribution_v1_3.html"
)

fig_driver_terms.write_html(
    "/content/custom_top_driver_terms_by_document_group_v1_3.html"
)

fig_slope.write_html(
    "/content/custom_slope_chart_safeguards_vs_innovation_v1_3.html"
)

if fig_slope_group_i is not None:
    fig_slope_group_i.write_html(
        "/content/custom_slope_chart_group_i_v1_3.html"
    )

if fig_slope_group_ii is not None:
    fig_slope_group_ii.write_html(
        "/content/custom_slope_chart_group_ii_v1_3.html"
    )

fig_doc_type.write_html(
    "/content/custom_document_type_comparison_v1_3.html"
)

print("Custom analysis outputs exported successfully.")


In [ ]:
# ============================================================
# Step 16 – Export supporting data tables for custom visuals
# ============================================================

heatmap_df.to_excel(
    "/content/custom_heatmap_data_v1_3.xlsx"
)

average_position_df.to_excel(
    "/content/custom_average_position_data_v1_3.xlsx",
    index=False
)

quadrant_df[
    [
        "document_id",
        "document",
        "jurisdiction",
        "plot_group",
        "document_type",
        "x_score",
        "y_score_adjusted",
        "quadrant"
    ]
].to_excel(
    "/content/custom_quadrant_assignment_v1_3.xlsx",
    index=False
)

quadrant_summary.to_excel(
    "/content/custom_quadrant_summary_v1_3.xlsx",
    index=False
)

top_driver_terms_df.to_excel(
    "/content/custom_top_driver_terms_by_document_group_v1_3.xlsx",
    index=False
)

slope_long_df.to_excel(
    "/content/custom_slope_chart_data_v1_3.xlsx",
    index=False
)

document_type_summary.to_excel(
    "/content/custom_document_type_summary_v1_3.xlsx",
    index=False
)

print("All supporting data tables exported successfully.")


In [ ]:
# ============================================================
# Step 16 – Check exported custom files
# ============================================================

for file in sorted(os.listdir("/content")):
    if file.startswith("custom_") and file.endswith((".xlsx", ".html", ".png")):
        print(file)


In [ ]:
# ============================================================
# Step 16 – Create ZIP package with all custom v1.3 outputs
# ============================================================

import zipfile
import os

zip_filename = "/content/custom_artefact_outputs_v1_3.zip"

files_to_zip = [
    "custom_corpus_results_v1_3.xlsx",
    "custom_corpus_summary_v1_3.xlsx",
    "custom_analytical_vocabulary_table_v1_3.xlsx",
    "custom_top20_analytical_vocabulary_by_document_v1_3.xlsx",
    "custom_regulatory_spectrum_v1_3.html",
    "custom_four_component_profile_v1_3.html",
    "custom_heatmap_v1_3.html",
    "custom_average_position_v1_3.html",
    "custom_quadrant_distribution_v1_3.html",
    "custom_top_driver_terms_by_document_group_v1_3.html",
    "custom_slope_chart_safeguards_vs_innovation_v1_3.html",
    "custom_slope_chart_group_i_v1_3.html",
    "custom_slope_chart_group_ii_v1_3.html",
    "custom_document_type_comparison_v1_3.html",
    "custom_heatmap_data_v1_3.xlsx",
    "custom_average_position_data_v1_3.xlsx",
    "custom_quadrant_assignment_v1_3.xlsx",
    "custom_quadrant_summary_v1_3.xlsx",
    "custom_top_driver_terms_by_document_group_v1_3.xlsx",
    "custom_slope_chart_data_v1_3.xlsx",
    "custom_document_type_summary_v1_3.xlsx"
]

with zipfile.ZipFile(zip_filename, "w") as zipf:
    for file in files_to_zip:
        file_path = f"/content/{file}"
        if os.path.exists(file_path):
            zipf.write(file_path, arcname=file)
        else:
            print(f"File not found and not added: {file}")

print("ZIP package created:", zip_filename)


In [ ]:
# ============================================================
# Step 26B – Download ZIP package
# ============================================================

from google.colab import files

files.download("/content/custom_artefact_outputs_v1_3.zip")
